# Free-GPU writing-feedback experiment

Select **Runtime → Change runtime type → T4 GPU**, then **Run all**. Requires your Google sign-in and an available Colab GPU. No purchases, provider API key, Drive mount, or publication.

This notebook contains a verified snapshot of the required training code; it does not clone a moving GitHub branch. It trains the small Qwen2.5-1.5B commentary adapter on historical CC-BY OpenStax excerpt/comment pairs. Amplify uploads, student review packets and private work are **not** embedded or used for training.

This is an experimental writing-commentary model, **not a trained six-category grader or narrative-scoring model**. Genuine teacher labels and held-out human review remain required for those tasks. Training completion is reported only after weights are saved. A checkpoint archive downloads before comparison inference, and a final archive includes held-out responses.


In [ ]:
import subprocess, pathlib, sys, os, io, zipfile, base64, hashlib, json
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → T4 GPU. No model download or training started.")
print("GPU:", torch.cuda.get_device_name(0))
repo = pathlib.Path("/content/essaygrader-colab-run") if pathlib.Path("/content").exists() else pathlib.Path.cwd() / "essaygrader-colab-run"
if repo.exists():
    raise RuntimeError("Run folder already exists. Download prior results, then use a fresh runtime.")
source_zip = base64.b64decode('...')
expected_source_sha256 = '4eff1ec631990edad8f1cb55dc641fde3230175862779d2eda50ea30d6e34301'
assert hashlib.sha256(source_zip).hexdigest() == expected_source_sha256, "Embedded training code checksum mismatch"
repo.mkdir()
with zipfile.ZipFile(io.BytesIO(source_zip)) as archive:
    for member in archive.infolist():
        target = (repo / member.filename).resolve()
        if not target.is_relative_to(repo.resolve()):
            raise RuntimeError("Invalid source archive path")
    archive.extractall(repo)
os.chdir(repo)
pathlib.Path("code-snapshot.json").write_text(json.dumps({"sha256": expected_source_sha256, "files": sorted(str(p.relative_to(repo)) for p in repo.rglob("*") if p.is_file())}, indent=2))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-training.txt"], check=True)
print("Training dependencies installed. Source snapshot:", expected_source_sha256)


## Collect the licensed teaching examples

The collector pins the historical edition, retains license/attribution and excludes quote-containing pairs conservatively. All excerpts from an author/essay remain in one split. The test split stays outside training. Do not replace the source with the current noncommercial edition.


In [ ]:
subprocess.run([sys.executable,"-m","model_work.openstax_annotations","download","--out","model_work/data/public/free-gpu-source"],check=True)
subprocess.run([sys.executable,"-m","model_work.openstax_annotations","prepare","--source","model_work/data/public/free-gpu-source","--out","model_work/prepared/free-gpu-commentary"],check=True)
subprocess.run([sys.executable,"-m","model_work.train_lora","--annotations","--prepared","model_work/prepared/free-gpu-commentary"],check=True)


## Resolve a model revision and train

The model identifier alone is not license proof. Resolve its exact revision, then the trainer verifies the pinned model card and Apache-2.0 LICENSE before loading weights. A missing or different license stops the run. One epoch and 4-bit QLoRA keep this an inexpensive experiment; free runtime timeouts may interrupt it.


In [ ]:
import json
from huggingface_hub import HfApi
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
revision = HfApi().model_info(model_id).sha
pathlib.Path("free-run-model.json").write_text(json.dumps({"model":model_id,"revision":revision},indent=2))
subprocess.run([sys.executable,"-m","model_work.train_lora","--annotations","--prepared","model_work/prepared/free-gpu-commentary","--train","--model",model_id,"--revision",revision,"--out","model_work/adapters/free-commentary-v1","--qlora","--epochs","1","--max-length","2048"],check=True)


In [ ]:
import shutil
checkpoint_dir=pathlib.Path("free-commentary-checkpoint")
checkpoint_dir.mkdir(exist_ok=False)
shutil.copytree("model_work/adapters/free-commentary-v1",checkpoint_dir/"training")
for filename in ("ATTRIBUTION.txt", "LICENSE", "manifest.json"):
    shutil.copy2(pathlib.Path("model_work/prepared/free-gpu-commentary")/filename,checkpoint_dir/filename)
for filename in ("free-run-model.json", "code-snapshot.json"):
    shutil.copy2(filename,checkpoint_dir/filename)
checkpoint_archive=shutil.make_archive("free-commentary-checkpoint","zip",checkpoint_dir)
print("Training finished; adapter saved. Quality review remains pending.")
try:
    from google.colab import files
    files.download(checkpoint_archive)
except ImportError:
    print("Checkpoint archive:",checkpoint_archive)


## Compare base and tuned commentary on unseen excerpts

This comparison is a review packet, not an automatic quality or grading pass. Read each answer against the original editorial comment. Check accuracy, useful explanation, unsupported claims and unnecessary rewriting. Both models receive the same input. Do not tune on these test examples and then call them an independent test.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from model_work.openstax_annotations import preflight, messages
_, splits = preflight("model_work/prepared/free-gpu-commentary")
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
base = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, trust_remote_code=False,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16),device_map="auto")
model = PeftModel.from_pretrained(base,"model_work/adapters/free-commentary-v1/adapter")
model.eval()
def response(row, tuned):
    inputs=tokenizer.apply_chat_template(messages(row)[:-1],add_generation_prompt=True,return_tensors="pt").to(model.device)
    with torch.inference_mode():
        if tuned:
            result=model.generate(inputs,attention_mask=torch.ones_like(inputs),max_new_tokens=192,do_sample=False,pad_token_id=tokenizer.eos_token_id)
        else:
            with model.disable_adapter():
                result=model.generate(inputs,attention_mask=torch.ones_like(inputs),max_new_tokens=192,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(result[0,inputs.shape[1]:],skip_special_tokens=True)
reviews=[]
for row in splits["test"]:
    reviews.append({"id":row["id"],"excerpt":row["excerpt"],"published_comment":row["comment"],
        "base_response":response(row,False),"tuned_response":response(row,True),"source_url":row["source_url"],
        "human_review":None})
pathlib.Path("free-commentary-review.json").write_text(json.dumps(reviews,indent=2,ensure_ascii=False))
print("Prepared",len(reviews),"unseen comparisons; human review remains pending.")


## Download your results

The adapter requires the same base model/revision to run. Keep attribution with the adapter. The download includes an unfilled human-review packet, license evidence and code/package versions. Nothing is installed into the live website automatically.


In [ ]:
import shutil
bundle=pathlib.Path("free-commentary-download");bundle.mkdir(exist_ok=False)
shutil.copytree("model_work/adapters/free-commentary-v1",bundle/"training")
for file in ("ATTRIBUTION.txt","LICENSE","manifest.json"):
    shutil.copy2(pathlib.Path("model_work/prepared/free-gpu-commentary")/file,bundle/file)
for file in ("free-run-model.json","free-commentary-review.json","code-snapshot.json"):
    shutil.copy2(file,bundle/file)
(bundle/"packages.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
(bundle/"README.txt").write_text("Experimental commentary adapter. Requires the pinned base model. No validated numeric grades or site deployment. Retain attribution and review held-out outputs before use.\n")
archive=shutil.make_archive("free-commentary-results","zip",bundle)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download from notebook files:",archive)
